# 4.0 — Modelos Pydantic sobre el grafo Neo4j

Ejercita los esquemas de `src/data/schemas/` alimentados desde el grafo de
farmacovigilancia en **Neo4j** (migración SQLite→Neo4j). El puente ya existe
en `src/data/enrichment.py`; aquí solo lo usamos y validamos.

Si Neo4j no está accesible o autenticado, el notebook **omite** las celdas de
BD con un mensaje y termina sin error.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent  # el notebook vive en notebooks/ -> raíz del repo
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from datetime import date

from neo4j.exceptions import AuthError, ServiceUnavailable

from src.config import neo4j_config
from src.data.enrichment import PharmacovigilanceStore, enrich_drug
from src.data.pharmacovigilance import graph
from src.data.repository import get_enriched_drug
from src.data.schemas import ATCCode, Drug, Interaction, Med, Patient, SideEffect

cfg = neo4j_config()
print(f"uri={cfg.uri}  user={cfg.user}  database={cfg.database}")  # nunca la contraseña

uri=bolt://localhost:7687  user=neo4j  database=neo4j


## Conexión y descubrimiento de un CID

Abre el `PharmacovigilanceStore`. Ante fallo de conexión o autenticación,
`STORE = None` y las celdas siguientes se omiten. Con conexión, busca un CID
de `:Drug` con al menos una arista `HAS_SIDE_EFFECT` (prefiere 33613,
amoxicilina; si no, el primero encontrado).

In [2]:
STORE: PharmacovigilanceStore | None = None
SAMPLE_CID: int | None = None

_DISCOVER = (
    "MATCH (d:Drug)-[:HAS_SIDE_EFFECT]->() "
    "RETURN d.cid AS cid ORDER BY (d.cid = 33613) DESC, d.cid LIMIT 1"
)

try:
    STORE = PharmacovigilanceStore(cfg)
    with STORE._driver.session(database=cfg.database) as _s:
        _rec = _s.run(_DISCOVER).single()
    SAMPLE_CID = _rec["cid"] if _rec else None
    if SAMPLE_CID is None:
        print("Neo4j conectado pero sin aristas HAS_SIDE_EFFECT — se omite la BD.")
    else:
        print(f"Conectado. CID de muestra = {SAMPLE_CID}")
except (ServiceUnavailable, AuthError) as exc:
    if STORE is not None:
        STORE.close()
    STORE = None
    print(f"Neo4j no disponible ({type(exc).__name__}) — se omiten las celdas de BD.")

Neo4j no disponible (AuthError) — se omiten las celdas de BD.
